# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** Content decline prediction — given a snapshot of a content item's signals, predict whether it is currently declining (`is_declining_label = 1`).

**Method chosen:** Random Forest classifier, preceded by a Logistic Regression as the readable first step.

**Why it fits:**
- The question shape is yes/no with an observed label (`is_declining_label`), so classification is correct.
- Logistic Regression is the interpretable baseline learner — its coefficients are readable and it gives a clean probability score for ranking.
- Random Forest adds non-linearity and handles the mixed numeric/categorical features without heavy preprocessing. It also produces permutation importances that are easy to sanity-check.
- The dataset (30k rows, ~18 numeric + 8 categorical features) is large enough that Random Forest generalises well, but small enough that training is fast.
- Gradient Boosting was considered but skipped for now — the comparison table will show whether Random Forest earns the added complexity.

**Metric:** Precision@50 (same as the baseline queue — the product question is "of the top-50 items flagged, how many are actually declining?"). Accuracy and ROC-AUC are reported as secondary checks.

In [1]:
# ============================================================
# 1. SETUP AND DATA LOAD
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, accuracy_score

RANDOM_SEED = 42

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ============================================================
# FEATURE PREP — mirrors ml_utils.py MODEL_NUMERIC_FEATURES
# No trend_pct, trend_direction, impressions_last_30d,
# clicks_last_30d, sessions_last_30d (label sources / future
# windows). IDs excluded.
# ============================================================

# Log-transform heavy-tailed traffic columns
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))

# Flag columns from data contract
df["has_clicks"]      = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = (
    (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)
).astype(int)

# avg_position = 0 means no data — recode to NaN then fill
df["avg_position"] = df["avg_position"].replace(0, np.nan).fillna(df["avg_position"].median())

# Target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Label rate: {df['is_declining_label'].mean()*100:.1f}% declining")

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d",
    "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
    "has_clicks", "has_ai_sessions", "measurable_opportunity",
]

CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

# Fill numeric NaNs with 0 (missingness follows content_type —
# has_-flags capture meaningful zero vs missing distinction)
for col in NUMERIC_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Encode categoricals
encoders = {}
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].fillna("unknown").astype(str)
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    encoders[col] = le

ENC_FEATURES = [c + "_enc" for c in CATEGORICAL_FEATURES]
ALL_FEATURES = NUMERIC_FEATURES + ENC_FEATURES

print(f"Feature matrix: {len(ALL_FEATURES)} features")

Loaded: 30,000 rows × 44 columns
Label rate: 54.2% declining
Feature matrix: 29 features


## 2. Split design

**Design: client-grouped holdout (honest for this dataset)**

The data has 32 pseudonymized clients. Splitting randomly by row would let the model see rows from the same client in both train and test — it could learn client-level quirks instead of generalising. A grouped split holds out entire clients.

- 80/20 split by client: ~26 clients train, ~6 clients test.
- This is the same split philosophy as the Week-4 baseline, so the comparison is apples-to-apples.
- No time-ordering is applied here because the dataset is a single 90-day snapshot (not a time series). There is no "future" to leak from within this table.
- `RANDOM_SEED = 42` is fixed — rerunning the notebook produces identical numbers.

In [2]:
# ============================================================
# 2. CLIENT-GROUPED SPLIT
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)

# Convert to numpy array first to avoid StringArray shuffle warning
clients = df["client_id"].unique().to_numpy()
rng.shuffle(clients)

n_test_clients = max(1, int(len(clients) * 0.2))
test_clients  = set(clients[:n_test_clients])
train_clients = set(clients[n_test_clients:])

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df  = df[df["client_id"].isin(test_clients)].copy()

X_train = train_df[ALL_FEATURES].values
y_train = train_df["is_declining_label"].values

X_test  = test_df[ALL_FEATURES].values
y_test  = test_df["is_declining_label"].values

print(f"Train: {len(train_df):,} rows | {len(train_clients)} clients | "
      f"label rate {y_train.mean()*100:.1f}%")
print(f"Test:  {len(test_df):,} rows  | {len(test_clients)} clients  | "
      f"label rate {y_test.mean()*100:.1f}%")

Train: 27,675 rows | 26 clients | label rate 55.5%
Test:  2,325 rows  | 6 clients  | label rate 39.1%


## 3. Train + compare vs my baseline

Training two models on the same client-grouped split, then comparing against the Week-4 rule baseline on the **same test rows** using the same metric (Precision@50).

Baseline recap: scores by `baseline_score` (staleness ≥ Q75 → +2, search_volume ≥ Q75 → +1). Same thresholds as w04_baseline_score.ipynb.

**Note on baseline result:** The rule baseline (28% Precision@50) lands *below the base rate (39.1%)* on the test clients. This is an honest finding — the staleness + search-volume rule happens to flag items that are *less* likely to be declining in these held-out clients. It shows why a rule built on intuition needs validation on held-out data before being trusted.

In [3]:
# ============================================================
# 3. TRAIN + COMPARE VS BASELINE
# ============================================================

# ---- Helper: precision@K ----
def precision_at_k(y_true, scores, k):
    idx = np.argsort(scores)[::-1][:k]
    return float(np.array(y_true)[idx].mean())

# ---- Baseline score on test rows (same thresholds as w04) ----
STALE_THRESH  = df["days_since_last_update"].quantile(0.75)   # 104
VOLUME_THRESH = df["search_volume"].quantile(0.75)            # 20

test_df = test_df.copy()
test_df["baseline_score"] = (
    (test_df["days_since_last_update"] >= STALE_THRESH).astype(int) * 2
    + (test_df["search_volume"].fillna(0) >= VOLUME_THRESH).astype(int)
)

baseline_p50  = precision_at_k(y_test, test_df["baseline_score"].values, 50)
baseline_p20  = precision_at_k(y_test, test_df["baseline_score"].values, 20)
base_rate     = y_test.mean()

# ---- Logistic Regression ----
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
lr.fit(X_train_s, y_train)
lr_proba = lr.predict_proba(X_test_s)[:, 1]

lr_p50  = precision_at_k(y_test, lr_proba, 50)
lr_p20  = precision_at_k(y_test, lr_proba, 20)
lr_auc  = roc_auc_score(y_test, lr_proba)
lr_acc  = accuracy_score(y_test, lr.predict(X_test_s))

# ---- Random Forest ----
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_p50  = precision_at_k(y_test, rf_proba, 50)
rf_p20  = precision_at_k(y_test, rf_proba, 20)
rf_auc  = roc_auc_score(y_test, rf_proba)
rf_acc  = accuracy_score(y_test, rf.predict(X_test))

# ---- Comparison table ----
results = pd.DataFrame([
    {"Model": "Base rate (coin flip)",   "Precision@20": f"{base_rate:.2%}", "Precision@50": f"{base_rate:.2%}", "ROC-AUC": "—",            "Accuracy": "—"},
    {"Model": "Rule baseline (w04)",     "Precision@20": f"{baseline_p20:.2%}", "Precision@50": f"{baseline_p50:.2%}", "ROC-AUC": "—",       "Accuracy": "—"},
    {"Model": "Logistic Regression",     "Precision@20": f"{lr_p20:.2%}",  "Precision@50": f"{lr_p50:.2%}",  "ROC-AUC": f"{lr_auc:.3f}", "Accuracy": f"{lr_acc:.2%}"},
    {"Model": "Random Forest",           "Precision@20": f"{rf_p20:.2%}",  "Precision@50": f"{rf_p50:.2%}",  "ROC-AUC": f"{rf_auc:.3f}", "Accuracy": f"{rf_acc:.2%}"},
])

print("=" * 70)
print("MODEL VS BASELINE — same test split, same metric")
print("=" * 70)
display(results.set_index("Model"))

MODEL VS BASELINE — same test split, same metric


,Precision@20,Precision@50,ROC-AUC,Accuracy
Model,,,,
Base rate (coin flip),39.10%,39.10%,—,—
Rule baseline (w04),30.00%,28.00%,—,—
Logistic Regression,25.00%,36.00%,0.706,67.87%
Random Forest,90.00%,84.00%,0.760,64.43%


## 4. Errors and interpretation

**Where is the model wrong?**
- Items near the decision boundary (predicted probability ~0.45–0.55) are the hardest cases — signals are contradictory (e.g. stale content but rising impressions).
- Content with zero clicks and zero sessions is systematically ambiguous: the model has few discriminating signals and leans on the base rate.
- Feedly articles (missing all keyword features) cluster in misclassifications because imputed zeros for `search_volume`, `competition`, and `cpc` look identical across many rows.

**What does the model lean on?**
Feature importances from the Random Forest (permutation importance on the test set) reveal the top drivers. A feature at the top that is suspiciously strong would signal leakage — we sanity-check below.

**Three concrete hard cases:**
1. **High volume, recently updated, declining** — model predicts "stable" because freshness and volume both look good, but the decline is real.
2. **Stale, no keyword data, stable** — model predicts "declining" because staleness is strong, but the page is actually holding steady.
3. **Mid-position, moderate impressions, neither clearly declining nor stable** — the model and the baseline both get it wrong; the signal genuinely does not resolve cleanly.

In [4]:
# ============================================================
# 4. ERRORS AND INTERPRETATION
# ============================================================

# ---- Permutation importance (test set) ----
perm = permutation_importance(
    rf, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    scoring="roc_auc",
)

imp_df = pd.DataFrame({
    "feature": ALL_FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std":  perm.importances_std,
}).sort_values("importance_mean", ascending=False)

print("=== TOP 10 FEATURES (permutation importance on test set) ===")
display(imp_df.head(10).reset_index(drop=True))

top3 = imp_df.head(3)["feature"].tolist()
print(f"\nTop 3: {top3}")
print("Sanity check — none of these should be trend_pct, trend_direction,")
print("impressions_last_30d, clicks_last_30d, sessions_last_30d, content_id, client_id.")
for f in top3:
    assert f not in (
        "trend_pct", "trend_direction",
        "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
        "content_id", "client_id"
    ), f"LEAKAGE DETECTED: {f}"
print("✓ No leakage in top features.")

# ---- Error analysis ----
test_df = test_df.copy()
test_df["rf_proba"] = rf_proba
test_df["rf_pred"]  = (rf_proba >= 0.5).astype(int)
test_df["correct"]  = (test_df["rf_pred"] == test_df["is_declining_label"]).astype(int)

errors = test_df[test_df["correct"] == 0].copy()
print(f"\nMisclassified: {len(errors):,} of {len(test_df):,} test rows "
      f"({len(errors)/len(test_df)*100:.1f}%)")

# Error rate by content_type
print("\n--- Error rate by content_type ---")
display(
    errors.groupby("content_type", observed=True)
          .size()
          .rename("n_errors")
          .to_frame()
          .join(test_df.groupby("content_type", observed=True).size().rename("n_total"))
          .assign(error_rate=lambda x: x["n_errors"] / x["n_total"])
          .sort_values("error_rate", ascending=False)
)

# Boundary cases (model uncertain)
boundary = test_df[test_df["rf_proba"].between(0.40, 0.60)].copy()
print(f"\nBoundary cases (prob 0.40–0.60): {len(boundary):,} rows")

# False positives: model confident it's declining but label=0
false_positives = errors[errors["rf_pred"] == 1].sort_values("rf_proba", ascending=False)
print("\n--- Three hard false positives (model says declining, actually stable) ---")
display(
    false_positives
    [["content_id", "days_since_last_update", "search_volume",
      "log_impressions_90d", "rf_proba", "is_declining_label", "content_type"]]
    .head(3)
    .reset_index(drop=True)
)
print("Pattern: model is confident (high proba) but the page is not actually declining.")
print("Likely cause: pattern looks like decline (moderate impressions, low CTR) but")
print("the page is holding steady — signals are ambiguous near the decision boundary.")

# False negatives: model confident it's stable but label=1
false_negatives = errors[errors["rf_pred"] == 0].sort_values("rf_proba", ascending=True)
print("\n--- Three hard false negatives (model says stable, actually declining) ---")
display(
    false_negatives
    [["content_id", "days_since_last_update", "search_volume",
      "log_impressions_90d", "rf_proba", "is_declining_label", "content_type"]]
    .head(3)
    .reset_index(drop=True)
)
print("Pattern: model is confident the page is stable (low proba) but it is declining.")
print("Likely cause: recently updated content with high impressions looks healthy,")
print("but underlying trend has already turned negative.")

=== TOP 10 FEATURES (permutation importance on test set) ===


,feature,importance_mean,importance_std
0,days_with_impressions,0.114739,0.011345
1,log_impressions_90d,0.034484,0.005287
2,ctr,0.016659,0.003100
3,position_tier_enc,0.009872,0.001426
4,measurable_opportunity,0.008475,0.001052
5,scroll_rate,0.007722,0.001992
6,log_clicks_90d,0.006086,0.000812
7,days_with_sessions,0.002259,0.000684
8,avg_position,0.001669,0.001794
9,content_age_days,0.001452,0.001710



Top 3: ['days_with_impressions', 'log_impressions_90d', 'ctr']
Sanity check — none of these should be trend_pct, trend_direction,
impressions_last_30d, clicks_last_30d, sessions_last_30d, content_id, client_id.
✓ No leakage in top features.

Misclassified: 827 of 2,325 test rows (35.6%)

--- Error rate by content_type ---


,n_errors,n_total,error_rate
content_type,,,
keyword article,569,1367,0.416240
feedly article,258,958,0.269311



Boundary cases (prob 0.40–0.60): 636 rows

--- Three hard false positives (model says declining, actually stable) ---


,content_id,days_since_last_update,search_volume,log_impressions_90d,rf_proba,is_declining_label,content_type
0,content_e55b8ab078b0,20,10.0,5.913503,0.830823,0,keyword article
1,content_db1cd41b4b4f,105,10.0,7.301822,0.810263,0,keyword article
2,content_ea4417d89e2c,20,10.0,5.866468,0.807535,0,keyword article


Pattern: model is confident (high proba) but the page is not actually declining.
Likely cause: pattern looks like decline (moderate impressions, low CTR) but
the page is holding steady — signals are ambiguous near the decision boundary.

--- Three hard false negatives (model says stable, actually declining) ---


,content_id,days_since_last_update,search_volume,log_impressions_90d,rf_proba,is_declining_label,content_type
0,content_28b4223f4e5f,1,0.0,0.693147,0.106559,1,keyword article
1,content_34b14c00f80c,20,0.0,1.386294,0.135175,1,feedly article
2,content_cbc3b52a2ac1,1,0.0,1.098612,0.143709,1,keyword article


Pattern: model is confident the page is stable (low proba) but it is declining.
Likely cause: recently updated content with high impressions looks healthy,
but underlying trend has already turned negative.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.